# Milestone 2: RAG Pipeline Exploration

Notebook for experimenting with the RAG pipeline components before integrating into the main codebase.

In [ ]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT))

from src.data_io import load_documents
from src.ranking import BM25Retriever, SemanticRetriever, ensure_search_text

docs = ensure_search_text(load_documents())
print(f"Loaded {len(docs)} documents")

## Semantic retrieval test

In [ ]:
semantic = SemanticRetriever(docs)
results = semantic.search("quiet dishwasher for apartment", top_k=5)
for r in results:
    print(f"{r.score:.4f} - {r.document.get('title', '')}")

## BM25 retrieval test

In [ ]:
bm25 = BM25Retriever(docs)
results = bm25.search("quiet dishwasher for apartment", top_k=5)
for r in results:
    print(f"{r.score:.4f} - {r.document.get('title', '')}")

## Hybrid retrieval (RRF)

In [ ]:
from src.hybrid_rag_pipeline import HybridDocumentRetriever, FusionConfig

hybrid = HybridDocumentRetriever(
    documents=docs, bm25=bm25, semantic=semantic,
    fusion=FusionConfig(mode="rrf", bm25_weight=0.4, semantic_weight=0.6)
)
results = hybrid.search("quiet dishwasher for apartment", top_k=5)
for r in results:
    print(f"{r.score:.4f} [{r.method}] - {r.document.get('title', '')}")

## LLM test with Ollama

In [ ]:
from src.llm_pipeline import OpenSourceChatModel

llm = OpenSourceChatModel(provider="ollama", model="qwen2.5:3b", temperature=0.2)
answer, _ = llm.chat([{"role": "user", "content": "What makes a good dishwasher?"}])
print(answer)

## Full RAG pipeline end-to-end

In [ ]:
from src.hybrid_rag_pipeline import HybridRAGPipeline

pipeline = HybridRAGPipeline(
    documents=docs, provider="ollama", model="qwen2.5:3b",
    default_k=5, bm25=bm25, semantic=semantic,
    fusion=FusionConfig(mode="rrf")
)
result = pipeline.answer("energy efficient dishwasher", k=5)
print("Answer:", result.answer[:500])

## Prompt variant comparison

In [ ]:
query = "best refrigerator water filter under 50 dollars"
for variant in ["strict", "concise", "analyst"]:
    result = pipeline.answer(query, k=5, prompt_variant=variant)
    print(f"=== {variant} ===")
    print(result.answer[:300])
    print()